# SFT Dataset Decontamination Check — against the already-published dataset

Loads your **already-built, already-pushed** `Ananda100/python-sft-dataset` directly
from Hugging Face and checks every one of its 280,317 examples against HumanEval and
MBPP test-set fingerprints, using the corrected (working) dataset loader IDs.

No need to re-run the full 5-source build pipeline — this works entirely from the
published dataset. Runtime: a few minutes (mostly downloading the SFT dataset once).

**Important:** the `normalize_for_dedup` function below is a standard reconstruction
(lowercase, collapse whitespace, strip punctuation). If your original build notebook
used a different normalization function, replace Cell 3 with your exact original
before trusting the result — using a different normalization than what built the
dataset could under- or over-count matches.


In [1]:
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00


In [2]:
# ======================= CONFIG =======================
SFT_REPO = "Ananda100/python-sft-dataset"   # your already-built, already-pushed SFT dataset
SPLIT    = "train"
OUT_JSON = "decontamination_check_result.json"
# ======================================================
print(f"Will check {SFT_REPO} [{SPLIT}] against HumanEval + MBPP")

Will check Ananda100/python-sft-dataset [train] against HumanEval + MBPP


## Normalization + fingerprint functions

In [3]:
import re
from datasets import load_dataset

def normalize_for_dedup(text):
    """Standard normalization: lowercase, collapse whitespace, strip punctuation,
    so near-identical problems match even with minor formatting differences.
    Replace this with your ORIGINAL build-pipeline version if it differs."""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def is_contaminated(record, contamination_set):
    key = normalize_for_dedup(record["problem"] + record["solution"])[:300]
    return key in contamination_set


print("Normalization functions ready")

Normalization functions ready


## Build the HumanEval + MBPP fingerprint set (corrected loader IDs)

In [4]:
def load_eval_contamination_set():
    fingerprints = set()

    try:
        he = load_dataset("openai/openai_humaneval", split="test")   # correct id (was: "openai_humaneval")
        for row in he:
            key = normalize_for_dedup(row.get("prompt", "") + row.get("canonical_solution", ""))
            fingerprints.add(key[:300])
        print(f"Loaded {len(he)} HumanEval problems")
    except Exception as e:
        print(f"WARNING: could not load HumanEval: {e}")

    try:
        mbpp = load_dataset("google-research-datasets/mbpp", "full", split="test")  # correct id+config (was: "mbpp")
        for row in mbpp:
            key = normalize_for_dedup(row.get("text", "") + row.get("code", ""))
            fingerprints.add(key[:300])
        print(f"Loaded {len(mbpp)} MBPP problems")
    except Exception as e:
        print(f"WARNING: could not load MBPP: {e}")

    print(f"Decontamination reference set built: {len(fingerprints)} fingerprints")
    return fingerprints


contamination_set = load_eval_contamination_set()
assert len(contamination_set) > 0, "Reference set is empty - loaders still failing, check error messages above"

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loaded 164 HumanEval problems


...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

Loaded 500 MBPP problems
Decontamination reference set built: 663 fingerprints


## Load the already-published SFT dataset and check every record

In [5]:
print(f"Loading {SFT_REPO} [{SPLIT}]...")
sft_ds = load_dataset(SFT_REPO, split=SPLIT)
print(f"Loaded {len(sft_ds)} SFT examples")

Loading Ananda100/python-sft-dataset [train]...


README.md:   0%|          | 0.00/4.29k [00:00<?, ?B/s]

datasetsforsft.jsonl: reconstructing file:   0%|          |  0.00B /  936MB            

datasetsforsft.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/280317 [00:00<?, ? examples/s]

Loaded 280317 SFT examples


In [6]:
from tqdm.auto import tqdm

contaminated_records = []
for i, record in enumerate(tqdm(sft_ds, desc="checking records")):
    if is_contaminated(record, contamination_set):
        contaminated_records.append(i)

print(f"\n{'='*60}")
print(f"RESULT: {len(contaminated_records)} / {len(sft_ds)} SFT examples "
      f"({100*len(contaminated_records)/len(sft_ds):.3f}%) match a HumanEval/MBPP fingerprint")
print(f"{'='*60}")

checking records:   0%|          | 0/280317 [00:00<?, ?it/s]


RESULT: 8 / 280317 SFT examples (0.003%) match a HumanEval/MBPP fingerprint


## Inspect matches (if any) and save the result

In [7]:
import json

if contaminated_records:
    print(f"\nContaminated record indices (first 20): {contaminated_records[:20]}")
    print("\nExample contaminated record:")
    r = sft_ds[contaminated_records[0]]
    print("Problem:", r["problem"][:300])
    print("Source:", r.get("source"))
    print("Layer:", r.get("layer"))
else:
    print("\nNo overlap found - the SFT dataset as released does not contain")
    print("HumanEval or MBPP problems by this fingerprint check.")

result = {
    "sft_repo": SFT_REPO,
    "n_sft_examples": len(sft_ds),
    "n_humaneval_mbpp_fingerprints": len(contamination_set),
    "n_contaminated": len(contaminated_records),
    "contamination_rate_pct": 100 * len(contaminated_records) / len(sft_ds),
    "contaminated_indices": contaminated_records[:100],  # cap for file size
}
with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2)
print(f"\nSaved: {OUT_JSON}")


Contaminated record indices (first 20): [207464, 208417, 208827, 208974, 211635, 216588, 218113, 224133]

Example contaminated record:
Problem: Write a python function to find the first position of an element in a sorted array.
Source: opencoder_sft_educational_instruct
Layer: diversity

Saved: decontamination_check_result.json


## Notes — how to use this result

- **N = 0**: decontamination claim is verified. Report in the paper as:
  *"Zero of 280,317 SFT examples matched a HumanEval or MBPP fingerprint under
  normalized problem+solution matching."*
- **N small (a handful)**: still a defensible release — report the exact count and,
  if you have time, regenerate the SFT set with those specific examples excluded
  before final release. Otherwise disclose the residual count honestly in Limitations.
- **N large**: stop and investigate — check a few `contaminated_records` examples by
  hand to confirm they're genuine matches (not just superficially similar problems
  like "reverse a string", which many datasets independently contain). If confirmed,
  the SFT set needs rebuilding without those examples, and any pass@1 evaluated
  before the fix should be treated as an upper bound, not the reported result.
- **If your original `normalize_for_dedup` differs from the reconstruction here**,
  replace Cell 3 (the normalization function) with your original before trusting
  the exact count — the direction of the finding (contaminated or not) is usually
  robust to normalization choice, but the exact N may shift slightly.
